# H3 — Per-anchor intervention strength

Test an anchor-to-eta lookup selected on calibration only. Types with at least two calibration landmarks choose eta 100 or 1000; unsupported types use the global scalar. Dome and minaret are fallback cases, not independently estimated effects.

**CPU report notebook.** Run `Point3_Shared54_Run.ipynb` for the experiments, then read its shared
results here. This notebook never trains a model, reruns a reconstruction, or selects settings
from confirmation scores. Default: 18 calibration and 36 confirmation landmarks, up to 50 eligible distinct photographs each.

Primary comparison: paired landmark difference in normalized Chamfer distance (structured − global
scalar); lower is better. Report the mean/median, wins/ties/losses and no-intervention comparison.
These 36 landmarks provide internal confirmation. The final 79-landmark holdout remains unrun.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from google.colab import drive
from IPython.display import display, Markdown, Image
drive.mount('/content/drive')
CALIBRATION_SIZE = 18
RUN_LABEL = 'v1'
OUTPUT = Path('/content/drive/MyDrive/Point3_Shared54/results') / f'shared54_83796943ffaf_cal{CALIBRATION_SIZE}_{RUN_LABEL}'
HYPOTHESIS = 'H3'
REPORT = OUTPUT/'reports'/HYPOTHESIS
if not (REPORT/'summary.json').exists():
    raise FileNotFoundError('Run the shared experiment notebook first; use its exact output path.')
display(Markdown((REPORT/'REPORT.md').read_text()))
print(json.dumps(json.loads((REPORT/'summary.json').read_text()), indent=2))


In [ ]:
paired = pd.read_csv(REPORT/'paired_results.csv')
display(paired)
plot = REPORT/'paired_differences.png'
if plot.exists() and len(paired): display(Image(filename=str(plot)))
if len(paired):
    print('Mean structured − no-intervention:', paired['structured_minus_baseline'].mean())
    if HYPOTHESIS == 'H3':
        display(paired.groupby(['anchor','anchor_fallback'], dropna=False)['structured_minus_scalar'].agg(['count','mean','median']))
        print('Identical H3/scalar predictions:', int(paired['same_prediction_as_scalar'].sum()))


In [ ]:
display(pd.read_csv(OUTPUT/'failures.csv'))
diagnostics = pd.read_csv(OUTPUT/'strength_diagnostics.csv')
display(diagnostics)
selection = OUTPUT/'selection.json'
if selection.exists(): print(json.dumps(json.loads(selection.read_text()), indent=2))
else: print('Calibration incomplete; no settings frozen yet.')
print('Raw metrics, prediction fingerprints and traces are in scenes/<scene hash>/<condition>.zip.')


## Interpretation and milestone use

Claim a benefit only for the evaluated cohort, report all failures, and distinguish a small paired
difference from robust evidence. A partial cohort is descriptive. A negative result is still useful:
it tests whether added structure earns its complexity. Review mask overlays and accepted-step traces.
The two-point candidate grids and one seed are deliberately limited preliminary experiments.
A larger frozen holdout evaluation and repeated seeds remain follow-up work.
